# Professional Data Analyst — Mini Project 1
## Advanced Python Data Exploration & Automated Reporting

**Dataset:** Sample - Superstore 2019.xls  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn

This notebook documents the full workflow required by the mini-project specification.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Import and Inspect the Dataset

In [ ]:
data = pd.read_excel("Sample - Superstore 2019.xls", sheet_name="Orders")

print("Shape:", data.shape)
display(data.head())
display(data.dtypes)
display(data.isna().sum())
print("Duplicate rows:", data.duplicated().sum())

## 3. Data Cleaning and Preprocessing
The cleaning step handles text consistency, dates, numeric values, missing values, duplicates, simple IQR outliers, and memory usage.

In [ ]:
# Remove extra spaces from text values
text_columns = data.select_dtypes(include="object").columns
for column in text_columns:
    data[column] = data[column].astype(str).str.strip()

# Convert dates and numeric columns
data["Order Date"] = pd.to_datetime(data["Order Date"], errors="coerce")
data["Ship Date"] = pd.to_datetime(data["Ship Date"], errors="coerce")
for column in ["Sales", "Profit", "Quantity", "Discount"]:
    data[column] = pd.to_numeric(data[column], errors="coerce")

# Handle missing values
for column in ["Sales", "Profit", "Quantity", "Discount"]:
    if data[column].isna().sum() > 0:
        data[column] = data[column].fillna(data[column].median())

for column in ["Order Date", "Ship Date"]:
    if data[column].isna().sum() > 0:
        data[column] = data[column].fillna(method="ffill")

# Remove duplicates
data = data.drop_duplicates()

# Simple outlier handling
for column in ["Sales", "Profit"]:
    q1 = data[column].quantile(0.25)
    q3 = data[column].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    data[column] = data[column].clip(lower, upper)

print("Shape after cleaning:", data.shape)

## 4. Feature Engineering

In [ ]:
data["Profit Margin"] = np.where(
    data["Sales"] != 0,
    data["Profit"] / data["Sales"],
    0
)

data["Shipping Duration"] = (
    data["Ship Date"] - data["Order Date"]
).dt.days

average_sales = data["Sales"].mean()
data["Sales Performance Category"] = np.where(
    data["Sales"] >= average_sales, "High", "Low"
)

data["Order Month"] = data["Order Date"].dt.to_period("M").astype(str)

display(data[[
    "Sales", "Profit", "Profit Margin",
    "Shipping Duration", "Sales Performance Category"
]].head())

## 5. Simple Memory Optimization

In [ ]:
data["Quantity"] = pd.to_numeric(data["Quantity"], downcast="integer")
data["Discount"] = pd.to_numeric(data["Discount"], downcast="float")
data["Sales"] = pd.to_numeric(data["Sales"], downcast="float")
data["Profit"] = pd.to_numeric(data["Profit"], downcast="float")

print(data.dtypes[["Sales", "Profit", "Quantity", "Discount"]])

## 6. Exploratory Analysis and KPI Summary

In [ ]:
sales_by_category = data.groupby("Category")["Sales"].sum().sort_values(ascending=False)
profit_by_region = data.groupby("Region")["Profit"].sum().sort_values(ascending=False)
monthly_sales = data.groupby("Order Month")["Sales"].sum()
sales_by_segment = data.groupby("Segment")["Sales"].sum()

kpis = {
    "Total Sales": data["Sales"].sum(),
    "Total Profit": data["Profit"].sum(),
    "Average Sales": data["Sales"].mean(),
    "Average Profit": data["Profit"].mean(),
    "Total Orders": data["Order ID"].nunique(),
    "Average Shipping Duration": data["Shipping Duration"].mean()
}

kpi_table = pd.DataFrame(list(kpis.items()), columns=["Metric", "Value"])
display(kpi_table)
display(sales_by_category)
display(profit_by_region)

## 7. Correlation Analysis

In [ ]:
numeric_data = data[[
    "Sales", "Quantity", "Discount", "Profit",
    "Profit Margin", "Shipping Duration"
]]
correlation = numeric_data.corr()
display(correlation)

## 8. Visualizations
The project requires a minimum of 8 visualizations.

### Visualization 1

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(sales_by_category.index, sales_by_category.values)
plt.title("Total Sales by Category")
plt.xlabel("Category"); plt.ylabel("Sales")
plt.tight_layout(); plt.show()

### Visualization 2

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(profit_by_region.index, profit_by_region.values)
plt.title("Total Profit by Region")
plt.xlabel("Region"); plt.ylabel("Profit")
plt.tight_layout(); plt.show()

### Visualization 3

In [ ]:
plt.figure(figsize=(11,5))
sns.lineplot(x=monthly_sales.index, y=monthly_sales.values, marker="o")
plt.title("Monthly Sales Trend")
plt.xticks(rotation=45)
plt.tight_layout(); plt.show()

### Visualization 4

In [ ]:
plt.figure(figsize=(7,7))
plt.pie(sales_by_category.values, labels=sales_by_category.index, autopct="%1.1f%%")
plt.title("Sales Share by Category")
plt.tight_layout(); plt.show()

### Visualization 5

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(data=data, x="Sales", y="Profit", hue="Category")
plt.title("Sales vs Profit")
plt.tight_layout(); plt.show()

### Visualization 6

In [ ]:
segment_plot = pd.DataFrame({"Segment": sales_by_segment.index, "Sales": sales_by_segment.values})
plt.figure(figsize=(8,5))
sns.barplot(data=segment_plot, x="Segment", y="Sales", color="steelblue")
plt.title("Sales by Customer Segment")
plt.tight_layout(); plt.show()

### Visualization 7

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(data=data, x="Profit", bins=30, kde=True, color="steelblue")
plt.title("Profit Distribution")
plt.tight_layout(); plt.show()

### Visualization 8

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(correlation, annot=True, fmt=".2f")
plt.title("Correlation Matrix")
plt.tight_layout(); plt.show()

## 9. Automated Export

In [ ]:
data.to_csv("cleaned_superstore.csv", index=False)
kpi_table.to_csv("kpi_summary.csv", index=False)
correlation.to_csv("correlation_matrix.csv")

with open("analytical_report.txt", "w", encoding="utf-8") as file:
    file.write("Superstore Analytical Report\n")
    file.write("===========================\n")
    for _, row in kpi_table.iterrows():
        file.write(f'{row["Metric"]}: {row["Value"]:.2f}\n')

print("Cleaned data, KPI summary, correlation matrix, and report exported.")

## 10. Conclusion

The workflow imports the required Superstore dataset, inspects it, performs cleaning and feature engineering, analyzes relationships and trends, creates at least eight visualizations, and exports analytical results automatically.